In [ ]:
### 1. 输出特征图尺寸

输入图像尺寸：\(3 \times 32 \times 32\)（通道数×高×宽）  
卷积核：16个，每个大小为 \(3 \times 5 \times 5\)  
填充（Padding）：2  
步幅（Stride）：2

输出高度和宽度计算公式：  
\[
H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} \right\rfloor + 1
\]  
代入数值：  
\[
H_{\text{out}} = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16
\]  
同理宽度 \(W_{\text{out}} = 16\)。  
输出通道数等于卷积核个数，即16。

因此，输出特征图尺寸为：  
\[
\boxed{16 \times 16 \times 16}
\]

### 2. 单个输出通道的一个像素值所需的乘法次数

每个输出像素由输入图像中对应感受野内的元素与卷积核对应元素逐点相乘后求和得到。  
一个卷积核的大小为 \(3 \times 5 \times 5\)，包含的元素个数为：  
\[
3 \times 5 \times 5 = 75
\]  
每个元素对应一次乘法操作（点乘），故单个输出像素需要 **75 次乘法**。

\[
\boxed{75}
\]

In [3]:
import numpy as np
from numpy.lib.stride_tricks import as_strided

def max_pool2d(input, kernel_size, stride=None, padding=0):
    """
    二维最大池化的 NumPy 实现（仅前向传播）
    
    参数:
        input: 4D 数组，形状 (batch_size, channels, height, width)
        kernel_size: int 或 tuple，池化窗口大小 (kh, kw)
        stride: int 或 tuple，步幅 (sh, sw)，默认等于 kernel_size
        padding: int 或 tuple，填充 (ph, pw)，默认 0
        
    返回:
        output: 池化后的数组，形状 (batch_size, channels, out_h, out_w)
    """
    # 解析参数为元组格式
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    
    if stride is None:
        sh = sw = kh
    elif isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding
    
    # 获取输入形状
    b, c, h, w = input.shape
    
    # 计算输出高度和宽度
    out_h = (h + 2 * ph - kh) // sh + 1
    out_w = (w + 2 * pw - kw) // sw + 1
    
    # 对输入进行填充（填充值为 -inf，确保填充区域不被选为最大值）
    pad_width = ((0, 0), (0, 0), (ph, ph), (pw, pw))
    padded = np.pad(input, pad_width, mode='constant', constant_values=-np.inf)
    
    # 使用 as_strided 创建滑动窗口视图
    # 计算原始数组的跨步
    s0, s1, s2, s3 = padded.strides
    # 新数组的跨步：(batch, channel, 行滑动, 列滑动, 窗口高, 窗口宽)
    new_strides = (s0, s1, sh * s2, sw * s3, s2, s3)
    # 新数组形状
    new_shape = (b, c, out_h, out_w, kh, kw)
    
    # 创建窗口视图（不复制数据）
    windows = as_strided(padded, shape=new_shape, strides=new_strides)
    
    # 沿窗口的高和宽维度取最大值
    output = np.max(windows, axis=(-2, -1))
    
    return output

# 创建一个随机输入 (batch=2, channels=3, height=32, width=32)
x = np.random.randn(2, 3, 32, 32)

# 池化：核大小 2x2，步幅 2，填充 0
y = max_pool2d(x, kernel_size=2, stride=2, padding=0)
print(y.shape)  # (2, 3, 16, 16)

# 非对称池化：核 3x5，步幅 (2,3)，填充 (1,2)
y2 = max_pool2d(x, kernel_size=(3,5), stride=(2,3), padding=(1,2))
print(y2.shape)  # (2, 3, 16, 11)

(2, 3, 16, 16)
(2, 3, 16, 11)


In [ ]:
### 1. 一个 \(5 \times 5\) 卷积层（不带偏置）的参数量

输入通道数 = \(C\)，输出通道数 = \(C\)，卷积核大小 = \(5 \times 5\)。  
参数量计算公式：  
\[
\text{参数量} = \text{输入通道数} \times \text{输出通道数} \times \text{核高} \times \text{核宽}
\]  
代入得：  
\[
\text{参数量} = C \times C \times 5 \times 5 = 25C^2
\]  
\[
\boxed{25C^2}
\]

### 2. 两个串联的 \(3 \times 3\) 卷积层（不带偏置，两层通道数都为 \(C\)）的总参数量

每个 \(3 \times 3\) 卷积层参数量为：  
\[
C \times C \times 3 \times 3 = 9C^2
\]  
两个串联的总参数量为：  
\[
2 \times 9C^2 = 18C^2
\]  
\[
\boxed{18C^2}
\]

**结论**：两个 \(3 \times 3\) 卷积级联（\(18C^2\)）相比单个 \(5 \times 5\) 卷积（\(25C^2\)）减少了约 28% 的参数量，同时具有相同的感受野（\(5 \times 5\)）。

In [5]:
import numpy as np

def conv2d(x, weight, bias=None, stride=1, padding=0):
    """
    手动实现二维卷积（支持多批量、多通道）
    
    参数:
        x: 输入 (batch, in_c, h, w)
        weight: 卷积核 (out_c, in_c, kh, kw)
        bias: 偏置 (out_c,) 或 None
        stride: 步幅 (int)
        padding: 填充 (int)
    
    返回:
        out: 输出 (batch, out_c, out_h, out_w)
    """
    batch, in_c, h, w = x.shape
    out_c, _, kh, kw = weight.shape
    
    # 填充
    pad_h, pad_w = padding, padding
    x_pad = np.pad(x, ((0, 0), (0, 0), (pad_h, pad_h), (pad_w, pad_w)), 
                   mode='constant', constant_values=0)
    
    # 计算输出尺寸
    out_h = (h + 2*padding - kh) // stride + 1
    out_w = (w + 2*padding - kw) // stride + 1
    
    out = np.zeros((batch, out_c, out_h, out_w))
    
    # 滑动窗口
    for b in range(batch):
        for oc in range(out_c):
            for i in range(out_h):
                for j in range(out_w):
                    # 窗口区域
                    h_start = i * stride
                    h_end = h_start + kh
                    w_start = j * stride
                    w_end = w_start + kw
                    window = x_pad[b, :, h_start:h_end, w_start:w_end]  # (in_c, kh, kw)
                    # 卷积操作（逐通道乘积累加）
                    val = np.sum(window * weight[oc])  # weight[oc] shape (in_c, kh, kw)
                    if bias is not None:
                        val += bias[oc]
                    out[b, oc, i, j] = val
    return out

def relu(x):
    """ReLU 激活函数"""
    return np.maximum(0, x)

class NiNBlock:
    """NiN 块的 NumPy 实现（仅前向传播）"""
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        # 第一个普通卷积层
        self.conv1_weight = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * 0.01
        self.conv1_bias = np.zeros(out_channels)
        self.conv1_stride = stride
        self.conv1_padding = padding
        
        # 第二个 1x1 卷积层
        self.conv2_weight = np.random.randn(out_channels, out_channels, 1, 1) * 0.01
        self.conv2_bias = np.zeros(out_channels)
        
        # 第三个 1x1 卷积层
        self.conv3_weight = np.random.randn(out_channels, out_channels, 1, 1) * 0.01
        self.conv3_bias = np.zeros(out_channels)
    
    def forward(self, x):
        # 普通卷积 + ReLU
        out = conv2d(x, self.conv1_weight, self.conv1_bias, 
                     stride=self.conv1_stride, padding=self.conv1_padding)
        out = relu(out)
        
        # 1x1 卷积 + ReLU
        out = conv2d(out, self.conv2_weight, self.conv2_bias, stride=1, padding=0)
        out = relu(out)
        
        # 第二个 1x1 卷积 + ReLU
        out = conv2d(out, self.conv3_weight, self.conv3_bias, stride=1, padding=0)
        out = relu(out)
        return out

# 使用示例
if __name__ == "__main__":
    # 创建一个 NiN 块：输入 3 通道，输出 32 通道，3x3 卷积，步幅 1，填充 1
    block = NiNBlock(3, 32, kernel_size=3, stride=1, padding=1)
    x = np.random.randn(1, 3, 224, 224)  # (batch, channel, height, width)
    y = block.forward(x)
    print(y.shape)  # 输出 (1, 32, 224, 224)

(1, 32, 224, 224)


In [ ]:
给定：  
\(x_1 = 2,\ x_2 = 4,\ x_3 = 6,\ x_4 = 8\)，  
\(\gamma = 2,\ \beta = 1,\ \epsilon = 0\)。

### 1. 计算批量均值和方差
\[
\mu = \frac{2+4+6+8}{4} = 5
\]
\[
\sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = \frac{9+1+1+9}{4} = 5
\]
\[
\sigma = \sqrt{5}
\]

### 2. 标准化（减去均值除以标准差）
\[
\hat{x}_1 = \frac{2-5}{\sqrt{5}} = -\frac{3}{\sqrt{5}},\quad 
\hat{x}_2 = \frac{4-5}{\sqrt{5}} = -\frac{1}{\sqrt{5}}
\]
\[
\hat{x}_3 = \frac{6-5}{\sqrt{5}} = \frac{1}{\sqrt{5}},\quad 
\hat{x}_4 = \frac{8-5}{\sqrt{5}} = \frac{3}{\sqrt{5}}
\]

### 3. 缩放和平移
\[
y_i = \gamma \hat{x}_i + \beta = 2\hat{x}_i + 1
\]
代入得：
\[
y_1 = 2\left(-\frac{3}{\sqrt{5}}\right) + 1 = 1 - \frac{6}{\sqrt{5}}
\]
\[
y_2 = 2\left(-\frac{1}{\sqrt{5}}\right) + 1 = 1 - \frac{2}{\sqrt{5}}
\]
\[
y_3 = 2\left(\frac{1}{\sqrt{5}}\right) + 1 = 1 + \frac{2}{\sqrt{5}}
\]
\[
y_4 = 2\left(\frac{3}{\sqrt{5}}\right) + 1 = 1 + \frac{6}{\sqrt{5}}
\]

### 最终结果
\[
\boxed{y_1 = 1 - \frac{6}{\sqrt{5}},\quad y_2 = 1 - \frac{2}{\sqrt{5}},\quad y_3 = 1 + \frac{2}{\sqrt{5}},\quad y_4 = 1 + \frac{6}{\sqrt{5}}}
\]

In [7]:
import numpy as np

def conv2d(x, weight, bias=None, stride=1, padding=0):
    batch, in_c, h, w = x.shape
    out_c, _, kh, kw = weight.shape
    # 填充
    x_pad = np.pad(x, ((0,0),(0,0),(padding,padding),(padding,padding)), mode='constant')
    out_h = (h + 2*padding - kh) // stride + 1
    out_w = (w + 2*padding - kw) // stride + 1
    out = np.zeros((batch, out_c, out_h, out_w))
    for b in range(batch):
        for oc in range(out_c):
            for i in range(out_h):
                for j in range(out_w):
                    h_start = i * stride
                    h_end = h_start + kh
                    w_start = j * stride
                    w_end = w_start + kw
                    window = x_pad[b, :, h_start:h_end, w_start:w_end]
                    out[b, oc, i, j] = np.sum(window * weight[oc])
                    if bias is not None:
                        out[b, oc, i, j] += bias[oc]
    return out

def batch_norm2d(x, gamma, beta, eps=1e-5):
    """简易 BN：对每个通道独立计算均值和方差，然后标准化"""
    batch, c, h, w = x.shape
    # 沿 (batch, h, w) 维度计算均值和方差
    mean = np.mean(x, axis=(0,2,3), keepdims=True)   # (1,c,1,1)
    var = np.var(x, axis=(0,2,3), keepdims=True)
    x_hat = (x - mean) / np.sqrt(var + eps)
    out = gamma.reshape(1,-1,1,1) * x_hat + beta.reshape(1,-1,1,1)
    return out

def relu(x):
    return np.maximum(0, x)

class Residual:
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        # 主路径：两个 3x3 卷积
        # conv1
        self.conv1_w = np.random.randn(out_channels, in_channels, 3, 3) * 0.01
        self.conv1_b = np.zeros(out_channels)
        self.bn1_gamma = np.ones(out_channels)
        self.bn1_beta = np.zeros(out_channels)
        # conv2
        self.conv2_w = np.random.randn(out_channels, out_channels, 3, 3) * 0.01
        self.conv2_b = np.zeros(out_channels)
        self.bn2_gamma = np.ones(out_channels)
        self.bn2_beta = np.zeros(out_channels)
        # 捷径
        self.use_1x1conv = use_1x1conv
        self.stride = stride
        if use_1x1conv:
            self.shortcut_conv_w = np.random.randn(out_channels, in_channels, 1, 1) * 0.01
            self.shortcut_conv_b = np.zeros(out_channels)
            self.shortcut_bn_gamma = np.ones(out_channels)
            self.shortcut_bn_beta = np.zeros(out_channels)
        else:
            self.shortcut_conv_w = None

    def forward(self, x):
        # 主路径
        out = conv2d(x, self.conv1_w, self.conv1_b, stride=self.stride, padding=1)
        out = batch_norm2d(out, self.bn1_gamma, self.bn1_beta)
        out = relu(out)
        out = conv2d(out, self.conv2_w, self.conv2_b, stride=1, padding=1)
        out = batch_norm2d(out, self.bn2_gamma, self.bn2_beta)

        # 捷径
        if self.use_1x1conv:
            shortcut = conv2d(x, self.shortcut_conv_w, self.shortcut_conv_b, stride=self.stride, padding=0)
            shortcut = batch_norm2d(shortcut, self.shortcut_bn_gamma, self.shortcut_bn_beta)
        else:
            shortcut = x  # 要求形状与 out 相同（通道、高、宽一致）
            # 如果 stride !=1，需要下采样（这里简化，仅报错提示）
            if self.stride != 1:
                raise ValueError("当 use_1x1conv=False 时，必须 stride=1 且 in_channels==out_channels 才能恒等映射")

        # 残差相加
        out = out + shortcut
        out = relu(out)
        return out

# 示例用法
if __name__ == "__main__":
    # 情况1：恒等映射（stride=1，相同通道）
    block = Residual(64, 64, stride=1, use_1x1conv=False)
    x = np.random.randn(4, 64, 32, 32)
    y = block.forward(x)
    print("输出形状（恒等映射）:", y.shape)  # (4, 64, 32, 32)

    # 情况2：下采样 + 1x1 卷积调整
    block2 = Residual(64, 128, stride=2, use_1x1conv=True)
    x2 = np.random.randn(4, 64, 32, 32)
    y2 = block2.forward(x2)
    print("输出形状（下采样）:", y2.shape)   # (4, 128, 16, 16)

输出形状（恒等映射）: (4, 64, 32, 32)
输出形状（下采样）: (4, 128, 16, 16)


In [ ]:
### 1. 为什么对底层特征提取层设置小学习率（或冻结），而对顶层输出层设置大学习率？

- **底层特征**（靠近输入的卷积层）在源数据集（如 ImageNet）上学习到的是**通用、可迁移的特征**（如边缘、纹理、颜色斑点等）。这些特征对于大多数视觉任务都是有效的，不应在微调时被剧烈改变，否则会破坏已经学到的良好表示。因此，使用较小的学习率（甚至完全冻结）可以保留这些通用特征，仅进行微小的调整。
- **顶层输出层**（靠近输出的全连接层或分类层）学习的是与源数据集**任务相关的特定特征**（如类别区分性特征）。在迁移到新数据集时，类别通常不同，因此该层需要**重新学习**或大幅调整。新初始化的顶层参数没有先验知识，需要用较大的学习率快速适应新数据集的分布，从而加快收敛并获得更好的性能。

### 2. 目标数据集非常小且与源数据集非常相似时的微调策略（防止过拟合）

- **冻结大部分底层特征提取层**：只对最后几层（或仅新添加的分类层）进行微调。因为底层特征已经足够通用，且小数据集无法提供足够的梯度信号来安全地更新大量参数。
- **使用较小学习率**：即使只微调顶层，也要使用比从头训练更小的学习率（例如，底层学习率为0，顶层学习率为源训练学习率的1/10或更小），避免剧烈更新破坏已有表示。
- **添加更强的正则化**：如增大权重衰减系数、使用Dropout（如果层中包含）、提前停止等。
- **数据增强**：对目标数据集应用丰富的数据增强（随机裁剪、翻转、颜色抖动等），以增加有效样本量。
- **采用特征提取器模式**：若目标数据集极小（每类仅几张图），完全冻结预训练网络，将其作为固定的特征提取器，仅训练一个新的线性分类器（如支持向量机或逻辑回归）。
- **减少模型容量**：如果仍过拟合，可考虑移除预训练网络的顶层，添加更小的全连接层（或全局平均池化+少量神经元），以减少可训练参数数量。

In [8]:
import numpy as np
import random
import math

def resize_bilinear(img, target_size):
    """
    双线性插值缩放图像
    img: (H, W, C)  numpy数组，值域任意
    target_size: (target_h, target_w)
    返回: (target_h, target_w, C)
    """
    src_h, src_w, c = img.shape
    dst_h, dst_w = target_size
    if src_h == dst_h and src_w == dst_w:
        return img.copy()
    # 计算采样位置
    y_ratio = src_h / dst_h
    x_ratio = src_w / dst_w
    y_idx = np.linspace(0, src_h - 1, dst_h)
    x_idx = np.linspace(0, src_w - 1, dst_w)
    # 网格化
    x_grid, y_grid = np.meshgrid(x_idx, y_idx)
    x0 = np.floor(x_grid).astype(np.int32)
    y0 = np.floor(y_grid).astype(np.int32)
    x1 = x0 + 1
    y1 = y0 + 1
    # 边界裁剪
    x0 = np.clip(x0, 0, src_w-1)
    x1 = np.clip(x1, 0, src_w-1)
    y0 = np.clip(y0, 0, src_h-1)
    y1 = np.clip(y1, 0, src_h-1)
    # 计算权重
    wa = (x1 - x_grid) * (y1 - y_grid)
    wb = (x_grid - x0) * (y1 - y_grid)
    wc = (x1 - x_grid) * (y_grid - y0)
    wd = (x_grid - x0) * (y_grid - y0)
    # 插值 (每个通道独立)
    img_flat = img.reshape(-1, c)
    idx_a = (y0 * src_w + x0).reshape(-1)
    idx_b = (y0 * src_w + x1).reshape(-1)
    idx_c = (y1 * src_w + x0).reshape(-1)
    idx_d = (y1 * src_w + x1).reshape(-1)
    out = (wa.reshape(-1,1) * img_flat[idx_a] +
           wb.reshape(-1,1) * img_flat[idx_b] +
           wc.reshape(-1,1) * img_flat[idx_c] +
           wd.reshape(-1,1) * img_flat[idx_d])
    return out.reshape(dst_h, dst_w, c)

def random_resized_crop(img, scale=(0.08, 1.0), size=(224, 224)):
    """
    随机裁剪并缩放到固定大小
    img: (H, W, C) numpy数组，值域任意
    返回: 裁剪并缩放后的图像 (size[0], size[1], C)
    """
    h, w, _ = img.shape
    area = h * w
    # 随机选择裁剪面积比例
    target_area = random.uniform(scale[0], scale[1]) * area
    # 随机宽高比（保持原始宽高比附近的随机性，这里简单采用原始宽高比）
    # 更精确的做法是允许宽高比在一定范围内变化，此处为了简洁，保持原始宽高比
    # 但为使增广更丰富，可以随机调整宽高比，我们实现一个常见做法：
    aspect_ratio = random.uniform(3/4, 4/3)  # 宽高比范围 0.75~1.33
    crop_w = int(round(math.sqrt(target_area * aspect_ratio)))
    crop_h = int(round(math.sqrt(target_area / aspect_ratio)))
    # 确保不越界
    if crop_w > w:
        crop_w = w
    if crop_h > h:
        crop_h = h
    # 随机起始点
    top = random.randint(0, h - crop_h)
    left = random.randint(0, w - crop_w)
    cropped = img[top:top+crop_h, left:left+crop_w, :]
    # 缩放到目标尺寸
    resized = resize_bilinear(cropped, size)
    return resized

def random_horizontal_flip(img, p=0.5):
    """以概率p水平翻转图像"""
    if random.random() < p:
        return np.flip(img, axis=1).copy()
    return img

def random_color_jitter(img, brightness=0.5, contrast=0.5, saturation=0.5):
    """
    随机调整亮度、对比度、饱和度
    img: (H, W, C) 值域 [0,1] 的 float 数组
    参数范围: 0.5 表示调整因子在 [0.5, 1.5] 之间
    """
    # 亮度: 乘随机因子
    brightness_factor = random.uniform(max(0, 1-brightness), 1+brightness)
    img = img * brightness_factor
    # 对比度: 以灰度均值中心缩放
    contrast_factor = random.uniform(max(0, 1-contrast), 1+contrast)
    gray_mean = np.mean(img, axis=(0,1), keepdims=True)  # (1,1,C)
    img = gray_mean + contrast_factor * (img - gray_mean)
    # 饱和度: 转换到 HSV 操作 S 通道
    # 先 RGB -> HSV，调整 S，再 HSV -> RGB
    saturation_factor = random.uniform(max(0, 1-saturation), 1+saturation)
    # RGB to HSV (简易版本)
    r, g, b = img[...,0], img[...,1], img[...,2]
    maxc = np.maximum(np.maximum(r, g), b)
    minc = np.minimum(np.minimum(r, g), b)
    delta = maxc - minc
    # 饱和度 S = delta / maxc  (当 maxc != 0)
    s = np.where(maxc != 0, delta / maxc, 0)
    # 调整饱和度
    s = np.clip(s * saturation_factor, 0, 1)
    # 重新计算 RGB (保持色调和亮度不变)
    # 公式: 当 S=0 时，RGB = maxc
    # 否则: R = maxc - delta * (maxc - r)/delta? 更简洁: 使用 HSV 转回 RGB
    # 简便方法: 使用开源实现
    # 这里实现标准转换:
    # 对于每个像素，色调 H 不变，亮度 V = maxc，饱和度变为新的 s
    # 输出 RGB = (V * (1 - s), V * (1 - s * (g-b)/delta?), 复杂)
    # 为了清晰，调用一个子函数
    img_hsv = rgb_to_hsv(img)
    img_hsv[..., 1] = np.clip(img_hsv[..., 1] * saturation_factor, 0, 1)
    img = hsv_to_rgb(img_hsv)
    # 最终裁剪到 [0,1]
    return np.clip(img, 0, 1)

def rgb_to_hsv(rgb):
    """rgb: (H,W,3) 值域 [0,1]"""
    r, g, b = rgb[...,0], rgb[...,1], rgb[...,2]
    maxc = np.maximum(np.maximum(r, g), b)
    minc = np.minimum(np.minimum(r, g), b)
    delta = maxc - minc
    h = np.zeros_like(maxc)
    # 计算色调
    mask = delta != 0
    r_eq_max = (r == maxc) & mask
    g_eq_max = (g == maxc) & mask
    b_eq_max = (b == maxc) & mask
    h[r_eq_max] = ((g[r_eq_max] - b[r_eq_max]) / delta[r_eq_max]) % 6
    h[g_eq_max] = ((b[g_eq_max] - r[g_eq_max]) / delta[g_eq_max]) + 2
    h[b_eq_max] = ((r[b_eq_max] - g[b_eq_max]) / delta[b_eq_max]) + 4
    h = h / 6.0  # 归一化到 [0,1]
    s = np.where(maxc != 0, delta / maxc, 0)
    v = maxc
    return np.stack([h, s, v], axis=-1)

def hsv_to_rgb(hsv):
    h, s, v = hsv[...,0], hsv[...,1], hsv[...,2]
    h = h * 6.0
    i = np.floor(h).astype(np.int32)
    f = h - i
    p = v * (1 - s)
    q = v * (1 - s * f)
    t = v * (1 - s * (1 - f))
    i_mod = i % 6
    rgb = np.zeros_like(hsv)
    # 每个通道选择
    mask0 = (i_mod == 0)
    rgb[mask0,0] = v[mask0]
    rgb[mask0,1] = t[mask0]
    rgb[mask0,2] = p[mask0]
    mask1 = (i_mod == 1)
    rgb[mask1,0] = q[mask1]
    rgb[mask1,1] = v[mask1]
    rgb[mask1,2] = p[mask1]
    mask2 = (i_mod == 2)
    rgb[mask2,0] = p[mask2]
    rgb[mask2,1] = v[mask2]
    rgb[mask2,2] = t[mask2]
    mask3 = (i_mod == 3)
    rgb[mask3,0] = p[mask3]
    rgb[mask3,1] = q[mask3]
    rgb[mask3,2] = v[mask3]
    mask4 = (i_mod == 4)
    rgb[mask4,0] = t[mask4]
    rgb[mask4,1] = p[mask4]
    rgb[mask4,2] = v[mask4]
    mask5 = (i_mod == 5)
    rgb[mask5,0] = v[mask5]
    rgb[mask5,1] = p[mask5]
    rgb[mask5,2] = q[mask5]
    return rgb

class ImageAugmentationPipeline:
    """图像增广管道 (纯 NumPy)"""
    def __init__(self, crop_scale=(0.08, 1.0), output_size=(224,224),
                 flip_prob=0.5, brightness=0.5, contrast=0.5, saturation=0.5):
        self.crop_scale = crop_scale
        self.output_size = output_size
        self.flip_prob = flip_prob
        self.brightness = brightness
        self.contrast = contrast
        self.saturation = saturation

    def __call__(self, img):
        """
        img: numpy array, shape (H,W,C), RGB顺序, 值域 [0,1] 或 [0,255]
        返回: numpy array, shape (C, H, W), 值域 [0,1], dtype=np.float32
        """
        # 确保值域为 [0,1]
        if img.dtype == np.uint8:
            img = img.astype(np.float32) / 255.0
        else:
            img = img.astype(np.float32)
        # 1. 随机裁剪缩放
        img = random_resized_crop(img, scale=self.crop_scale, size=self.output_size)
        # 2. 随机水平翻转
        img = random_horizontal_flip(img, p=self.flip_prob)
        # 3. 随机颜色抖动
        img = random_color_jitter(img, brightness=self.brightness,
                                  contrast=self.contrast, saturation=self.saturation)
        # 4. 转换为 CHW 格式 (C, H, W)
        img = np.transpose(img, (2, 0, 1))
        return img

# ===================== 使用示例 =====================
if __name__ == "__main__":
    # 生成一张模拟图像 (300, 400, 3) 随机像素
    fake_image = np.random.randint(0, 256, (300, 400, 3), dtype=np.uint8)
    pipeline = ImageAugmentationPipeline()
    augmented = pipeline(fake_image)
    print("增广后形状:", augmented.shape)   # (3, 224, 224)
    print("值域:", augmented.min(), augmented.max())

增广后形状: (3, 224, 224)
值域: 0.0 1.0


In [ ]:
### 已知边界框
- 真实框 \( A = [10, 10, 50, 50] \)
- 预测框 \( B = [30, 30, 70, 70] \)

### 1. 计算交集区域
交集左上角坐标：
\[
x_{\text{left}} = \max(10, 30) = 30, \quad y_{\text{top}} = \max(10, 30) = 30
\]
交集右下角坐标：
\[
x_{\text{right}} = \min(50, 70) = 50, \quad y_{\text{bottom}} = \min(50, 70) = 50
\]
交集宽度和高度：
\[
w_{\text{inter}} = 50 - 30 = 20, \quad h_{\text{inter}} = 50 - 30 = 20
\]
交集面积：
\[
S_{\text{inter}} = 20 \times 20 = 400
\]

### 2. 计算各自面积
\[
S_A = (50 - 10) \times (50 - 10) = 40 \times 40 = 1600
\]
\[
S_B = (70 - 30) \times (70 - 30) = 40 \times 40 = 1600
\]

### 3. 计算并集面积
\[
S_{\text{union}} = S_A + S_B - S_{\text{inter}} = 1600 + 1600 - 400 = 2800
\]

### 4. 计算 IoU
\[
\text{IoU} = \frac{S_{\text{inter}}}{S_{\text{union}}} = \frac{400}{2800} = \frac{1}{7} \approx 0.142857
\]

### 最终结果
\[
\boxed{\dfrac{1}{7}}
\]

In [10]:
import numpy as np

def label_smoothing_cross_entropy_np(logits, labels, epsilon=0.1, reduction='mean'):
    """
    标签平滑交叉熵损失的 NumPy 实现
    
    参数:
        logits: 模型输出 (batch_size, num_classes)，任意实数
        labels: 真实类别索引 (batch_size,)
        epsilon: 平滑因子，默认 0.1
        reduction: 'mean' 或 'sum'
    
    返回:
        损失值（标量）
    """
    batch_size, num_classes = logits.shape
    
    # 1. Softmax 得到概率分布（数值稳定）
    max_logits = np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits - max_logits)
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    
    # 2. Log 概率
    log_probs = np.log(probs + 1e-8)   # 微小值防止 log(0)
    
    # 3. 构建平滑目标分布
    smooth_target = np.full_like(logits, epsilon / (num_classes - 1))
    smooth_target[np.arange(batch_size), labels] = 1 - epsilon
    
    # 4. 交叉熵损失： -sum(target * log_probs)
    losses = -np.sum(smooth_target * log_probs, axis=1)
    
    if reduction == 'mean':
        return np.mean(losses)
    elif reduction == 'sum':
        return np.sum(losses)
    else:
        return losses

# ========== 示例运行并打印输出 ==========
if __name__ == "__main__":
    # 2个样本，3个类别
    logits = np.array([[2.0, 1.0, 0.1],
                       [0.5, 2.5, 0.3]])
    labels = np.array([0, 1])
    epsilon = 0.1
    
    loss = label_smoothing_cross_entropy_np(logits, labels, epsilon=epsilon)
    print(f"logits:\n{logits}")
    print(f"labels: {labels}")
    print(f"标签平滑交叉熵损失 (ε={epsilon}): {loss:.6f}")

logits:
[[2.  1.  0.1]
 [0.5 2.5 0.3]]
labels: [0 1]
标签平滑交叉熵损失 (ε=0.1): 0.496040
